# `long` / `wide` — Pure 1-to-1 Reshaping

`long` melts numeric columns to rows. `wide` spreads a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.pt.long(c='variable', v='value', id_vars=None)` | `c`, `v`, `id_vars` | wide → long (melt) |
| `.pt.wide(c='variable', v='value', index=None)` | `c`, `v`, `index` (or `r`) | long → wide (pure 1:1 unmelt) |

They are designed to work together as a **strictly complementary 1-to-1 reshaping pair**.

> **Note on Aggregation:**
> `wide()` is strictly for 1-to-1 structural reshaping without aggregation. If your data contains duplicate row/column keys and requires summarization (`mean`, `sum`, `count`/`n`), use [**`pt.pivot`**](pivot.ipynb).

In [1]:
import pytae as pt
import pandas as pd

penguins = pt.sample("penguins")
flights = pt.sample("flights")
healthexp = pt.sample("healthexp")

flights.head()

,year,month,passengers
0,1949,Jan,112
1,1949,Feb,118
2,1949,Mar,132
3,1949,Apr,129
4,1949,May,121


## 1. `long` — Melt Columns to Rows

Melts all numeric columns to rows by default, preserving non-numeric columns as row identifiers:

In [2]:
# Melt numeric columns into (variable, value) pairs
penguins.pt.select("species", "bill_length_mm", "bill_depth_mm").pt.long().head()

,species,variable,value
0,Adelie,bill_length_mm,39.1
1,Adelie,bill_length_mm,39.5
2,Adelie,bill_length_mm,40.3
3,Adelie,bill_length_mm,NaN
4,Adelie,bill_length_mm,36.7


In [3]:
# Customize target column names for variable and value
flights.pt.long(c="metric", v="value").head()

,month,metric,value
0,Jan,year,1949
1,Feb,year,1949
2,Mar,year,1949
3,Apr,year,1949
4,May,year,1949


In [4]:
# Explicit id_vars: specify which columns remain fixed
healthexp.pt.long(id_vars=["Year", "Country"], c="Metric", v="Amount").head()

,Year,Country,Metric,Amount
0,1970,Germany,Spending_USD,252.311
1,1970,France,Spending_USD,192.143
2,1970,Great Britain,Spending_USD,123.993
3,1970,Japan,Spending_USD,150.437
4,1970,USA,Spending_USD,326.961


## 2. `wide` — Pure 1-to-1 Unmelting

Pivots categorical values in column `c` into column headers, pulling cell values from `v`. Every `(index, c)` combination must be unique.

In [5]:
# flights has exactly one row per year + month: perfect 1-to-1 unmelt
flights_wide = flights.pt.wide(c="month", v="passengers")
flights_wide.head()

,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [6]:
# Explicit index parameter (r= / index=)
demo_long = pd.DataFrame({
    "id": [1, 2, 3],
    "quarter": ["Q1", "Q2", "Q3"],
    "sales": [100, 200, 300]
})
demo_long.pt.wide(r="id", c="quarter", v="sales")

,id,Q1,Q2,Q3
0,1,100.0,NaN,NaN
1,2,NaN,200.0,NaN
2,3,NaN,NaN,300.0


## 3. Strict 1-to-1 Reshaping Guarantees

If your DataFrame contains duplicate index/column pairs, `wide()` refuses to silently aggregate and raises a clear error directing you to `pt.pivot()`:

In [7]:
# penguins has multiple individuals per (island, species), so wide() halts:
try:
    penguins.pt.wide(index="island", c="species", v="body_mass_g")
except ValueError as e:
    print("Error caught successfully:")
    print(e)

Error caught successfully:
wide() encountered duplicate entries for index ['island'] and column 'species'. wide() is strictly for 1-to-1 reshaping without aggregation. Use pt.pivot() / -pivot to summarize or aggregate duplicate entries.


To summarize duplicate keys across dimensions, use [`pt.pivot`](pivot.ipynb) instead:

In [8]:
# Proper aggregation with pt.pivot:
penguins.pt.pivot(r="island", c="species", v="body_mass_g", a="mean")

,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


## 4. Round-Trip Reshaping (`long` ↔ `wide`)

Because `long` and `wide` are strict bijections, a DataFrame can be unpivoted to long format and seamlessly restored to its original structure:

In [9]:
# Start with wide flights
wide_orig = flights.pt.wide(c="month", v="passengers")

# 1. Melt to long
long_form = wide_orig.pt.long(c="month", v="passengers", id_vars=["year"])

# 2. Reshape back to wide
wide_restored = long_form.pt.wide(r="year", c="month", v="passengers")

# 3. Verify exact match (aligning column order)
pd.testing.assert_frame_equal(wide_orig[wide_restored.columns], wide_restored)
print("Round-trip verified: wide_orig matches wide_restored perfectly!")
wide_restored.head()

Round-trip verified: wide_orig matches wide_restored perfectly!


,year,Apr,Aug,Dec,Feb,Jan,Jul,Jun,Mar,May,Nov,Oct,Sep
0,1949,129,148,118,118,112,148,135,132,121,104,119,136
1,1950,135,170,140,126,115,170,149,141,125,114,133,158
2,1951,163,199,166,150,145,199,178,178,172,146,162,184
3,1952,181,242,194,180,171,230,218,193,183,172,191,209
4,1953,235,272,201,196,196,264,243,236,229,180,211,237
